In [1]:
# clone repo
!git clone https://github.com/abramovgeorge/hse-dla-asr.git /content/hse-dla-asr

In [3]:
# prepare environment
%cd /content/hse-dla-asr
!pip install -q uv gdown
!uv venv --python 3.10 /content/asr-env
!uv pip install --python /content/asr-env/bin/python -r requirements.txt

import os

os.environ["MPLBACKEND"] = "Agg"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

/content/hse-dla-asr
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 104.2 MB/s eta 0:00:00
Using CPython 3.10.22
Creating virtual environment at: /content/asr-env
Activate with: source /content/asr-env/bin/activate
Using Python 3.10.22 environment at: /content/asr-env
Resolved 152 packages in 2.51s
Prepared 152 packages in 1m 03s
Installed 152 packages in 793ms
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + async-timeout==5.0.1
 + attrs==26.1.0
 + audiomentations==0.43.1
 + audioread==3.1.0
 + black==26.10.1
 + certifi==2026.10.18
 + cffi==2.1.1
 + cfgv==3.5.0
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + comet-ml==3.58.8
 + configobj==5.0.9
 + contourpy==1.3.2
 + cwevals==0.0.1b1
 + cycler==0.12.1
 + datasets==3.6.0
 + decorator==5.3.1
 + dill==0.3.8
 + distlib==0.4.4
 + distro==1.9.0
 + dulwich==1.2.17
 + everett==3.1.0
 + exceptiongroup==1.3.1
 + filelo

In [4]:
# download model parts
from pathlib import Path
import shutil
import gdown

MODEL_ID = "1o4tfb3WCsIrpPvNtz4IlOKk3E5Jzbnjj"

gdown.download(
    id=MODEL_ID,
    output="/content/asr_model.zip",
    fuzzy=True,
)

shutil.unpack_archive("/content/asr_model.zip", "/content/asr_model")

model_dir = Path("/content/asr_model")
weights_path = next(model_dir.rglob("model_best.pth"))
train_config_path = next(model_dir.rglob("config.yaml"))
tokenizer_path = next(model_dir.rglob("bpe_128.json"))

Downloading...
From (original): https://drive.google.com/uc?id=1o4tfb3WCsIrpPvNtz4IlOKk3E5Jzbnjj
From (redirected): https://drive.google.com/uc?id=1o4tfb3WCsIrpPvNtz4IlOKk3E5Jzbnjj&confirm=t&uuid=ece16603-c683-4d3e-8c52-0c3e25795a77
To: /content/asr_model.zip
100%|██████████| 213M/213M [00:00<00:00, 223MB/s]


In [5]:
# download dataset
DATASET_ID = "1tPDxP64TGFerexuX7E2tJYbvQnYYCXZh" # id (the last hash in the link)

gdown.download(
    id=DATASET_ID,
    output="/content/asr_dataset.zip",
    fuzzy=True,
)

dataset_dir = Path("/content/asr_dataset")
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

shutil.unpack_archive("/content/asr_dataset.zip", dataset_dir)

dataset_root = next(
    path.parent
    for path in dataset_dir.rglob("audio")
    if path.is_dir()
)

audio_dir = dataset_root / "audio"
transcription_dir = dataset_root / "transcriptions"

Downloading...
From: https://drive.google.com/uc?id=1tPDxP64TGFerexuX7E2tJYbvQnYYCXZh
To: /content/asr_dataset.zip
100%|██████████| 929k/929k [00:00<00:00, 130MB/s]


In [6]:
# prepare inference config
import yaml

with train_config_path.open() as f:
    config = yaml.safe_load(f)

config["datasets"] = {
    "test": {
        "_target_": "src.datasets.CustomDirAudioDataset",
        "audio_dir": str(audio_dir),
        "transcription_dir": (
            str(transcription_dir)
            if transcription_dir.is_dir()
            else None
        ),
        "instance_transforms": "${transforms.instance_transforms.inference}",
    }
}

config["metrics"] = {"inference": []}

audio_count = sum(
    path.is_file() and path.suffix in (".wav", ".flac", ".mp3", ".m4a")
    for path in audio_dir.iterdir()
)
config["dataloader"]["batch_size"] = min(16, audio_count)
config["dataloader"]["num_workers"] = 0

config["text_encoder"]["tokenizer_path"] = str(tokenizer_path)
config["text_encoder"]["tokenizer_text_dir"] = None
config["text_encoder"]["beam_search_njobs"] = 2
config["text_encoder"]["lm_model"]["device"] = "cuda"

config["inferencer"] = {
    "device": "cuda",
    "device_tensors": ["spectrogram", "text_encoded"],
    "seed": 1,
    "from_pretrained": str(weights_path),
    "save_path": "/content/asr_predictions",
    "decoding": "beam_search",
}

with open("src/configs/demo_inference.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

In [7]:
# inference
!/content/asr-env/bin/python -u inference.py --config-name=demo_inference






ConformerCTC(
  (encoder): ConformerEncoder(
    (downsampler): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): LeakyReLU(negative_slope=0.01)
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): LeakyReLU(negative_slope=0.01)
    )
    (downsample_mlp): Linear(in_features=1024, out_features=256, bias=True)
    (layers): ModuleList(
      (0-11): 12 x ConformerLayer(
        (ffn1): Sequential(
          (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=256, out_features=1024, bias=True)
          (2): SiLU()
          (3): Dropout(p=0.1, inplace=False)
          (4): Linear(in_features=1024, out_features=256, bias=True)
          (5): Dropout(p=0.1, inplace=False)
        )
        (ffn2): Sequential(
          (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=256, out_features=1024, bias=True)
          (2): S

In [8]:
# calculate metrics
import subprocess

predictions_dir = Path("/content/asr_predictions/test")

if transcription_dir.is_dir():
    result = subprocess.run(
        [
            "/content/asr-env/bin/python",
            "-m", "scripts.calc_metrics",
            "--target-dir", str(transcription_dir),
            "--predicted-dir", str(predictions_dir),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("No transcriptions")

No transcriptions


In [9]:
# download preds
from google.colab import files

shutil.make_archive(
    "/content/asr_predictions",
    "zip",
    "/content/asr_predictions",
)

files.download("/content/asr_predictions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>